In [ ]:
import pandas as pd
import itertools
from pymatgen.core import Composition, Element
from collections import Counter, defaultdict

In [ ]:
peroxidase = pd.read_excel('./data/peroxidase.xlsx')

In [ ]:
peroxidase.year.value_counts().sort_index(ascending=False)

In [ ]:
metal_counter = Counter()
nonmetal_counter = Counter()

for f in peroxidase['formula'].dropna():
    comp = Composition(f)
    metals_in_formula = set()
    nonmetals_in_formula = set()
    for el in comp.get_el_amt_dict().keys():
        if Element(el).is_metal:
            metals_in_formula.add(el)
        else:
            nonmetals_in_formula.add(el)
    for m in metals_in_formula:
        metal_counter[m] += 1
    for n in nonmetals_in_formula:
        nonmetal_counter[n] += 1

metal_freq = pd.DataFrame.from_dict(metal_counter, orient='index', columns=['count']).sort_values('count', ascending=False)
nonmetal_freq = pd.DataFrame.from_dict(nonmetal_counter, orient='index', columns=['count']).sort_values('count', ascending=False)

In [ ]:
metal_freq.head(50)

In [ ]:
nonmetal_freq.head(50)

In [ ]:
def get_crystal_system(symbol: str) -> str:
    if symbol.startswith('c'):
        return 'Cubic'
    elif symbol.startswith('hR'):
        return 'Trigonal'
    elif symbol.startswith('hP'):
        return 'Hexagonal'
    elif symbol.startswith(('tP', 'tI', 'tC')):
        return 'Tetragonal'
    elif symbol.startswith('o'):
        return 'Orthorhombic'
    elif symbol.startswith('m'):
        return 'Monoclinic'
    elif symbol.startswith('a'):
        return 'Triclinic'
    else:
        return None

peroxidase['pearson'] = peroxidase['pearson'].astype(str).apply(get_crystal_system).str.lower()

In [ ]:
peroxidase.pearson.value_counts()

In [ ]:
elem_to_set = defaultdict(set)

for idx, f in enumerate(peroxidase['formula'].dropna()):
    comp = Composition(f)
    for el in comp.get_el_amt_dict().keys():
        elem_to_set[el].add(idx)
        
pairs = []
for a, b in itertools.combinations(elem_to_set.keys(), 2):
    set_a, set_b = elem_to_set[a], elem_to_set[b]
    inter = len(set_a & set_b)
    union = len(set_a | set_b)
    if union > 0:
        jaccard = inter / union
        pairs.append((a, b, jaccard, inter, union))

jac_df = pd.DataFrame(pairs, columns=['elem1', 'elem2', 'jaccard', 'intersection', 'union'])
jac_df_filtered = jac_df[(jac_df['intersection'] >= 5) &
                         (jac_df['union'] >= 10)].copy()
jac_df_filtered['pair'] = jac_df_filtered['elem1'] + "-" + jac_df_filtered['elem2']

jac_df_filtered[['pair', 'jaccard', 'intersection', 'union']] \
            .sort_values('jaccard', ascending=False) \
            .head(15)